# Генератор рукописных строк — примеры

Шрифты должны лежать в `assets/fonts/` (`python scripts/fetch_fonts.py`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from IPython.display import display
from src.synth import HandwrittenLineGenerator, make_generator

FONTS = str(ROOT / 'assets' / 'fonts')

## Сэмплим пары (текст, картинка)

`sample()` возвращает `(PIL.Image, str)` — тугой кроп строки на бумаге (короткая
сторона 224, натуральный аспект, без белых полей). `text_dirs` — список папок с `.txt`
(обходятся рекурсивно и случайно). Пустой список — встроенный словарь.

In [ ]:
gen = HandwrittenLineGenerator.from_dirs(text_dirs=[], font_dirs=FONTS, curriculum=False)

for i in range(5):
    img, text = gen.sample(make_generator(42, 0, i))
    print(text)
    display(img)

## Свои тексты + длина строк + переносы

- `len_chars=(мин, макс)` — длина строки в символах; разная длина -> разный размер картинки.
- `p_hyphenate` — доля строк, оканчивающихся переносом `-` на середине слова.
  Перенос работает только на тексте из `text_dirs` (бегущий текст), не на встроенном словаре.

In [ ]:
gen = HandwrittenLineGenerator.from_dirs(
    text_dirs=['D:/corpora/books', 'D:/corpora/notes'],   # ваши папки с .txt
    font_dirs=FONTS,
    len_chars=(15, 45),    # длина строки в символах
    p_hyphenate=0.3,       # ~30% строк с переносом '-'
    curriculum=False,      # полная длина сразу (иначе короче на ранних step)
)
print('файлов в корпусе:', len(gen.sampler._files))

for i in range(6):
    img, text = gen.sample(make_generator(1, 0, i))
    print(text)
    display(img)

## В обучении (на лету)

Генератор бесконечный — оборачиваем в `IterableDataset` с пер-воркерным сидом.
Здесь `curriculum` включён (по умолчанию), сложность растёт со `step`.

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader

class SynthLines(IterableDataset):
    def __init__(self, gen, base_seed=42):
        self.gen, self.base_seed = gen, base_seed
    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        wid = info.id if info else 0
        i = 0
        while True:
            img, text = self.gen.sample(make_generator(self.base_seed, wid, i), step=i)
            yield {'image': img, 'text': text}
            i += 1

gen = HandwrittenLineGenerator.from_dirs(text_dirs=[], font_dirs=FONTS)
loader = DataLoader(SynthLines(gen), batch_size=4, num_workers=0,
                    collate_fn=lambda b: ([x['image'] for x in b], [x['text'] for x in b]))
images, texts = next(iter(loader))
print(texts)
# дальше: processor(images=images).pixel_values + tokenizer(texts) -> TrOCR

Подсказки: `step` в `sample(rng, step)` управляет сложностью (0 — просто, `warmup_steps` — полная);
короткая сторона картинки = `output.min_side` (224), аспект натуральный; для квадратного входа
TrOCR оберни в `fit_to_square(img, 384)`; больше шрифтов — в `assets/fonts/`.